# 第17讲：整理连接诊断技能库

练习保护应急技能、原生状态管理、模型自主整理与恢复。所有技能和现场数据均为构造教学数据。题面、运行和停止方式见 [README](README.md)。请使用仓库固定依赖的 Python 内核，每轮重新启动内核；本 Notebook 不读取个人 Hermes Home。


In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd()
if not (HERE / 'curation_workshop.py').is_file():
    HERE = HERE / 'examples/17-curator'
assert (HERE / 'curation_workshop.py').is_file(), '请从本讲或仓库根目录打开'
sys.path.insert(0, str(HERE))
from curation_workshop import CurationLab, tree
lab = CurationLab()
lab.seed()
print('本轮目录：', lab.root)


## 1. 先观察状态，不请求模型

`seed()` 通过原生 `skill_manage` 创建四项技能，并构造活动时间、保护应急手册。下面直接调用原生自动转换；核对 active、stale、archived 与 pinned 的关系。


In [ ]:
from agent import curator, curator_backup
from tools import skill_usage

counts = curator.apply_automatic_transitions(now=lab.clock)
lab.observe_transitions(counts)
{n: skill_usage.get_record(n) for n in ('connection-pool-triage', 'connection-leak-investigation', 'disaster-recovery', 'retired-db-debug')}


## 2. 由 Curator 自己判断

下一格会调用 GLM。`prepare_review()` 建立快照、安装课程读取记录适配器，并限制工具与迭代次数；不指定合并目标，不替换原生整理提示。缺少凭据时隐藏询问。失败不是 no-op，保持原样也不是失败。


In [ ]:
lab.prepare_review()
try:
    returned = curator.run_curator_review(consolidate=True, synchronous=True)
    lab.observe_review(returned)
finally:
    lab._restore_observer()
lab.native['outcome']


## 3. 文件少了不等于能力保留

分别验证容量压力与疑似连接泄漏。执行函数在新会话中使用如下真实调用，并在 `finally` 关闭 Agent 与 SessionDB；模型只有读取技能权限：

```python
agent = AIAgent(model='glm-5.2', provider='custom', api_mode='chat_completions',
    api_key=os.environ['GLM_API_KEY'], base_url=os.environ['GLM_BASE_URL'],
    enabled_toolsets=['skills'], max_iterations=8, max_tokens=2048,
    request_overrides={'temperature': 0}, quiet_mode=True, skip_context_files=True,
    load_soul_identity=False, save_trajectories=False, session_id='probe-' + kind, session_db=db)
```

完整调用与结果检查在 [curation_workshop.py](curation_workshop.py) 的 `probe()`；诊断及下一步说明必须来自模型，不能从整理数量推断。程序检查结构、诊断类别和真实支持文件读取；解释质量由你阅读。


In [ ]:
for kind in ('pool', 'leak'):
    print(kind, lab.probe(kind))


## 4. 单项恢复与全库回滚

`restore_skill(name)` 恢复某个归档技能；`curator_backup.rollback(snapshot_name)` 恢复整份快照。下面只在本轮教学副本中删除一个支持文件，检查失联，再恢复到整理前。即使模型选择 no-op，也能使用第一步归档的退役技能完成恢复练习。


In [ ]:
archived = skill_usage.list_archived_skill_names()
if archived:
    print(skill_usage.restore_skill(archived[0]))
lab.recover()
result = lab.finish()
result


## 继续思考

哪些低频能力应保护？什么情况下不应合并？如何保留专用脚本及其引用？支持文件存在是否足以证明任务仍可完成？这些是设计判断，没有自动化唯一答案。本 Notebook 不验证周期调度或生产目录采用。
